# Module 01 — LLM Basics (GitHub Models)

**Goal:** Make your first LLM API call and understand every part of it.

This notebook uses **[GitHub Models](https://github.com/marketplace/models)** — a free, hosted inference API for prototyping. No GPU, no install, just a GitHub account.

It is the OpenAI-compatible mirror of `notebook_ollama.ipynb`. Every cell does the same thing — only the `BASE_URL`, `API_KEY` and `MODEL` change.

By the end you will know:
- What `messages` are and why roles matter
- What the raw API response object looks like
- Why we set `temperature=0` for deterministic tasks like SQL generation

---
**Run each cell in order (Shift+Enter).**

## Prerequisites — Get a GitHub token

You only need to do this once.

**1. Create a GitHub Personal Access Token (PAT)**

Go to https://github.com/settings/tokens and create a **fine-grained token** (or classic — both work). The free GitHub Models tier requires no special scope: a token with default permissions is enough.

**2. Export it in your shell** (or put it in `.env` at the repo root):

```bash
export GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

Or in `.env`:
```
GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

**3. (Optional) Pick a model.** The default below is `openai/gpt-4o-mini`. Browse the full catalog at https://github.com/marketplace?type=models — popular options include `openai/gpt-4o`, `meta/Meta-Llama-3.1-70B-Instruct`, `mistral-ai/Mistral-Nemo`.

> Free tier has rate limits — a few requests per minute and a daily cap. Plenty for learning.

In [ ]:
# Verify the token is present before we start
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))

token = os.getenv("GITHUB_TOKEN") or os.getenv("LLM_API_KEY")
if token:
    print(f"✓ Token found ({len(token)} chars, starts with {token[:6]}…)")
else:
    print("✗ No GITHUB_TOKEN set. See the prerequisites cell above.")

## Step 1 — Setup

GitHub Models exposes an **OpenAI-compatible API** at `https://models.github.ai/inference`.
That means the exact same Python code works with GitHub Models, OpenAI, Ollama, Groq, or any other provider —
you just change the `base_url` and `model` name.

| Provider | `base_url` | `api_key` | `model` |
|----------|-----------|----------|--------|
| **GitHub Models (free)** | `https://models.github.ai/inference` | your GitHub PAT | `"openai/gpt-4o-mini"` |
| OpenAI | `https://api.openai.com/v1` | your OpenAI key | `"gpt-4o-mini"` |
| Ollama (local) | `http://localhost:11434/v1` | `"ollama"` | `"llama3.2:1b"` |
| Groq | `https://api.groq.com/openai/v1` | your Groq key | `"llama-3.1-8b-instant"` |

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load .env from the repo root (two levels up) if it exists
load_dotenv(Path("../../.env"))

# Defaults to GitHub Models — override by setting these in your .env
BASE_URL = os.getenv("LLM_BASE_URL", "https://models.github.ai/inference")
API_KEY  = os.getenv("LLM_API_KEY")  or os.getenv("GITHUB_TOKEN")
MODEL    = os.getenv("LLM_MODEL",    "openai/gpt-4o-mini")

if not API_KEY:
    raise RuntimeError("Set GITHUB_TOKEN (or LLM_API_KEY) in your environment or .env file.")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

print(f"Model:    {MODEL}")
print(f"Endpoint: {BASE_URL}")

## Step 2 — Your first API call

The minimum required fields are `model` and `messages`.

`messages` is a list of conversation turns. Each turn has:
- `role`: who is speaking — `"system"`, `"user"`, or `"assistant"`
- `content`: what they said

We'll start with just a `user` message — no system prompt yet.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is 2 + 2? Answer in one word."}
    ],
)

# The model's answer is nested at this path:
print(response.choices[0].message.content)

## Step 3 — Inspect the full response

There's a lot more in the response than just the text.
Let's look at the fields that matter.

In [ ]:
print("=== Full response object ===")
print(response)
print()
print("=== Fields you care about ===")
print(f"model used:      {response.model}")
print(f"finish_reason:   {response.choices[0].finish_reason}")
print(f"content:         {response.choices[0].message.content}")
print(f"role:            {response.choices[0].message.role}")
print(f"prompt tokens:   {response.usage.prompt_tokens}")
print(f"response tokens: {response.usage.completion_tokens}")

**`finish_reason`** tells you *why* the model stopped generating:
- `"stop"` — normal end of response
- `"length"` — hit the `max_tokens` limit
- `"tool_calls"` — the model wants to call a function (Module 03!)

In Module 04 (the agentic loop), we check `finish_reason` on every turn
to know whether the agent is done or still working.

> **Note on `gpt-4o-mini`:** This is a small, fast model — perfect for learning and prototyping.
> When you need more capability, swap to `openai/gpt-4o` or one of the larger Llama variants — the API code stays identical.

## Step 4 — The system prompt

The `system` role gives the model persistent instructions that apply to every user message.
It's the most powerful knob for controlling model behavior.

In [ ]:
def ask(question: str, system: str = "") -> str:
    """Helper: send a question and return the response text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


# Without a system prompt:
print("No system prompt:")
print(ask("Tell me about Paris."))
print()

# With a system prompt that constrains the output:
print("With system prompt (one sentence only):")
print(ask(
    "Tell me about Paris.",
    system="You are a tour guide. Answer every question in exactly one sentence."
))

## Step 5 — Temperature: determinism vs creativity

`temperature=0` always picks the most likely next token → same question, same answer every time.  
`temperature=1` samples from the distribution → same question, different answers each run.

For SQL generation we always use `temperature=0`.  
For creative tasks (brainstorming, summarization) you'd use 0.7–1.0.

> **Tip:** Run the cell below multiple times to see the effect.
>
> **Heads-up:** the free GitHub Models tier rate-limits requests. If you see a 429, wait a few seconds and retry.

In [ ]:
question = "Name one random country."

print("temperature=0 (deterministic) — should be the same every run:")
for _ in range(3):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    print(" ", r.choices[0].message.content)

print()
print("temperature=1 (creative) — should vary:")
for _ in range(3):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=1,
    )
    print(" ", r.choices[0].message.content)

## Step 6 — Multi-turn conversation

LLMs are **stateless** — they have no memory between API calls.
You simulate memory by appending every turn to `messages` before the next call.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_message: str) -> str:
    """Add a user message, get a response, append both to history."""
    messages.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    assistant_message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_message})
    return assistant_message


print("Turn 1:", chat("My name is Alice."))
print()
print("Turn 2:", chat("What is my name?"))  # should remember 'Alice'
print()
print(f"Messages in history: {len(messages)} (grows with every turn!)")

## Step 7 — Swap to a different model (optional)

Try a different GitHub Models entry, a paid OpenAI model, or a local Ollama model — the code doesn't change at all.

In [ ]:
# A bigger GitHub-hosted model
# MODEL_BIG = "openai/gpt-4o"
# r = client.chat.completions.create(model=MODEL_BIG, messages=[{"role": "user", "content": "Hi!"}])

# A Llama on GitHub Models
# MODEL_LLAMA = "meta/Meta-Llama-3.1-70B-Instruct"

# Local Ollama (see notebook_ollama.ipynb)
# client_ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# OpenAI cloud (paid)
# client_openai = OpenAI(base_url="https://api.openai.com/v1", api_key=os.getenv("OPENAI_API_KEY"))

# All of the above use the SAME API call:
# response = client_XYZ.chat.completions.create(model=MODEL_XYZ, messages=[...])

print("Uncomment one of the blocks above and re-run to compare outputs across providers.")

## Summary

| Concept | Key point |
|---------|----------|
| `messages` | The full conversation history sent on every call |
| `system` role | Persistent instructions that shape every response |
| `finish_reason` | Why the model stopped — important in the agentic loop (Module 04) |
| `temperature=0` | Deterministic output — use this for SQL generation |
| Stateless | LLMs don't remember — you must append history yourself |
| Provider-agnostic | Same Python code works with GitHub Models, OpenAI, Ollama, Groq |

**Next:** Module 02 — how to get reliable *structured* output (JSON) from an LLM instead of free-form text.